# THEMIS-Aの電磁場データについて、full orbit (8 Hz)とparticle burst (512 Hz)、low telemetry (16 Hz)とhigh telemetry (128 Hz)の両方を用いる

# FACの定義の付加

# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# THEMIS-Aの電場・磁場データのdownload

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/20:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330'

psp.themis.fgm(trange=time_range, probe='a', level='l2', no_update=True, get_support_data=True)                 # fgh: 128 Hz, fgl: 16 Hz, fgs: 2.74 sec
psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efp', no_update=True, get_support_data=True) # efp: 512 Hz
psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efi', no_update=True, get_support_data=True) # eff: 8 Hz
print("--- Loaded tplot variables ---")
print(pt.tplot_names())

In [ ]:
Espin_data_gsm  = pt.data_quants['tha_efs_dot0_gsm']
E8_data_gsm     = pt.data_quants['tha_eff_dot0_gsm']
E512_data_gsm   = pt.data_quants['tha_efp_gsm']
Bspin_data_gsm  = pt.data_quants['tha_fgs_gsm']
B16_data_gsm    = pt.data_quants['tha_fgl_gsm']
B128_data_gsm   = pt.data_quants['tha_fgh_gsm']

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
Espin_data_gsm  = Espin_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
E8_data_gsm     = E8_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
E512_data_gsm   = E512_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
Bspin_data_gsm  = Bspin_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
B16_data_gsm    = B16_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
B128_data_gsm   = B128_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')

In [ ]:
import xarray as xr
import numpy as np

def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [ ]:
ds_Espin_data_gsm = xr.Dataset(
    data_vars={
        "Espin_gsm_x": (("time",), Espin_data_gsm[:, 0].values),
        "Espin_gsm_y": (("time",), Espin_data_gsm[:, 1].values),
        "Espin_gsm_z": (("time",), Espin_data_gsm[:, 2].values),
    },
    coords={
        "time": Espin_data_gsm["time"].values,
    },
)
ds_E8_data_gsm = xr.Dataset(
    data_vars={
        "E8_gsm_x": (("time",), E8_data_gsm[:, 0].values),
        "E8_gsm_y": (("time",), E8_data_gsm[:, 1].values),
        "E8_gsm_z": (("time",), E8_data_gsm[:, 2].values),
    },
    coords={
        "time": E8_data_gsm["time"].values,
    },
)
ds_E512_data_gsm = xr.Dataset(
    data_vars={
        "E512_gsm_x": (("time",), E512_data_gsm[:, 0].values),
        "E512_gsm_y": (("time",), E512_data_gsm[:, 1].values),
        "E512_gsm_z": (("time",), E512_data_gsm[:, 2].values),
    },
    coords={
        "time": E512_data_gsm["time"].values,
    },
)
ds_Bspin_data_gsm = xr.Dataset(
    data_vars={
        "Bspin_gsm_x": (("time",), Bspin_data_gsm[:, 0].values),
        "Bspin_gsm_y": (("time",), Bspin_data_gsm[:, 1].values),
        "Bspin_gsm_z": (("time",), Bspin_data_gsm[:, 2].values),
    },
    coords={
        "time": Bspin_data_gsm["time"].values,
    },
)
ds_B16_data_gsm = xr.Dataset(
    data_vars={
        "B16_gsm_x": (("time",), B16_data_gsm[:, 0].values),
        "B16_gsm_y": (("time",), B16_data_gsm[:, 1].values),
        "B16_gsm_z": (("time",), B16_data_gsm[:, 2].values),
    },
    coords={
        "time": B16_data_gsm["time"].values,
    },
)
ds_B128_data_gsm = xr.Dataset(
    data_vars={
        "B128_gsm_x": (("time",), B128_data_gsm[:, 0].values),
        "B128_gsm_y": (("time",), B128_data_gsm[:, 1].values),
        "B128_gsm_z": (("time",), B128_data_gsm[:, 2].values),
    },
    coords={
        "time": B128_data_gsm["time"].values,
    },
)

In [ ]:
def uniq_and_sort_time(ds):
    idx = ds.get_index('time')
    mask = ~idx.duplicated()          # 最初の出現だけ True
    return ds.isel(time=mask).sortby('time')

In [ ]:
ds_Espin_data_gsm   = uniq_and_sort_time(ds_Espin_data_gsm)
ds_E8_data_gsm      = uniq_and_sort_time(ds_E8_data_gsm)
ds_E512_data_gsm    = uniq_and_sort_time(ds_E512_data_gsm)
ds_Bspin_data_gsm   = uniq_and_sort_time(ds_Bspin_data_gsm)
ds_B16_data_gsm     = uniq_and_sort_time(ds_B16_data_gsm)
ds_B128_data_gsm    = uniq_and_sort_time(ds_B128_data_gsm)

In [ ]:
ds_Espin_data_gsm_segs = split_by_gap(ds_Espin_data_gsm, gap_thr=np.timedelta64(11, 's'))
for ds_ in ds_Espin_data_gsm_segs:
    print(ds_.time)

In [ ]:
ds_E8_data_gsm_segs = split_by_gap(ds_E8_data_gsm, gap_thr=np.timedelta64(500, 'ms'))
for ds_ in ds_E8_data_gsm_segs:
    print(ds_.time)

In [ ]:
ds_E512_data_gsm_segs = split_by_gap(ds_E512_data_gsm, gap_thr=np.timedelta64(8, 'ms'))
for ds_ in ds_E512_data_gsm_segs:
    print(ds_.time)

In [ ]:
ds_Bspin_data_gsm_segs = split_by_gap(ds_Bspin_data_gsm, gap_thr=np.timedelta64(11, 's'))
for ds_ in ds_Bspin_data_gsm_segs:
    print(ds_.time)

In [ ]:
ds_B16_data_gsm_segs = split_by_gap(ds_B16_data_gsm, gap_thr=np.timedelta64(250, 'ms'))
for ds_ in ds_B16_data_gsm_segs:
    print(ds_.time)

In [ ]:
ds_B128_data_gsm_segs = split_by_gap(ds_B128_data_gsm, gap_thr=np.timedelta64(250, 'ms'))
for ds_ in ds_B128_data_gsm_segs:
    print(ds_.time)

# 電場データと磁場データの時間を合わせる

In [ ]:
def make_ds_EB_func(ds_E, E_vars, ds_B, B_vars, time_base, output_vars):
    ds_E_interp = ds_E.interp(time=time_base, method='linear')
    ds_B_interp = ds_B.interp(time=time_base, method='linear')

    da_Ex   = ds_E_interp[E_vars[0]]
    da_Ey   = ds_E_interp[E_vars[1]]
    da_Ez   = ds_E_interp[E_vars[2]]
    da_Bx   = ds_B_interp[B_vars[0]]
    da_By   = ds_B_interp[B_vars[1]]
    da_Bz   = ds_B_interp[B_vars[2]]

    ds_EB   = xr.Dataset({
        output_vars[0]: da_Ex,
        output_vars[1]: da_Ey,
        output_vars[2]: da_Ez,
        output_vars[3]: da_Bx,
        output_vars[4]: da_By,
        output_vars[5]: da_Bz,
    })

    ds_EB   = ds_EB.dropna(dim='time', how='any')

    return ds_EB

In [ ]:
Espin_vars  = ['Espin_gsm_x', 'Espin_gsm_y', 'Espin_gsm_z']
E8_vars     = ['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z']
E512_vars   = ['E512_gsm_x', 'E512_gsm_y', 'E512_gsm_z']
Bspin_vars  = ['Bspin_gsm_x', 'Bspin_gsm_y', 'Bspin_gsm_z']
B16_vars    = ['B16_gsm_x', 'B16_gsm_y', 'B16_gsm_z']
B128_vars   = ['B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z']
EBspin_vars = ['Espin_gsm_x', 'Espin_gsm_y', 'Espin_gsm_z', 'Bspin_gsm_x', 'Bspin_gsm_y', 'Bspin_gsm_z']
EB8_vars    = ['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z', 'B8_gsm_x', 'B8_gsm_y', 'B8_gsm_z']
EB128_vars  = ['E128_gsm_x', 'E128_gsm_y', 'E128_gsm_z', 'B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z']

In [ ]:
ds_EBspin_gsm_segs  = []

ds_EBspin_gsm_segs.append(make_ds_EB_func(ds_Espin_data_gsm_segs[0], Espin_vars, ds_Bspin_data_gsm_segs[0], Bspin_vars, ds_Espin_data_gsm_segs[0].time, EBspin_vars).dropna(dim='time', how='all'))

print(ds_EBspin_gsm_segs)

In [ ]:
ds_EB8_gsm_segs = []

ds_EB8_gsm_segs.append(make_ds_EB_func(ds_E8_data_gsm_segs[0], E8_vars, ds_B16_data_gsm_segs[0], B16_vars, ds_E8_data_gsm_segs[0].time, EB8_vars).dropna(dim='time', how='all'))
ds_EB8_gsm_segs.append(make_ds_EB_func(ds_E8_data_gsm_segs[1], E8_vars, ds_B16_data_gsm_segs[2], B16_vars, ds_E8_data_gsm_segs[1].time, EB8_vars).dropna(dim='time', how='all'))
ds_EB8_gsm_segs.append(make_ds_EB_func(ds_E8_data_gsm_segs[2], E8_vars, ds_B16_data_gsm_segs[2], B16_vars, ds_E8_data_gsm_segs[2].time, EB8_vars).dropna(dim='time', how='all'))

print(ds_EB8_gsm_segs)

In [ ]:
ds_EB128_gsm_segs = []

ds_EB128_gsm_segs.append(make_ds_EB_func(ds_E512_data_gsm_segs[0], E512_vars, ds_B128_data_gsm_segs[0], B128_vars, ds_B128_data_gsm_segs[0].time, EB128_vars).dropna(dim='time', how='all'))
ds_EB128_gsm_segs.append(make_ds_EB_func(ds_E512_data_gsm_segs[1], E512_vars, ds_B128_data_gsm_segs[1], B128_vars, ds_B128_data_gsm_segs[1].time, EB128_vars).dropna(dim='time', how='all'))
ds_EB128_gsm_segs.append(make_ds_EB_func(ds_E512_data_gsm_segs[2], E512_vars, ds_B128_data_gsm_segs[2], B128_vars, ds_B128_data_gsm_segs[2].time, EB128_vars).dropna(dim='time', how='all'))

print(ds_EB128_gsm_segs)

In [ ]:
path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330'
os.makedirs(path_base_save_plot, exist_ok=True)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['Espin_gsm_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['Espin_gsm_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['Espin_gsm_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['Bspin_gsm_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['Bspin_gsm_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['Bspin_gsm_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (GSM)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (GSM)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (GSM)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (GSM)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (GSM)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (GSM)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_gsm_spin'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_gsm_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EBspin_gsm_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E8_gsm_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E8_gsm_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E8_gsm_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B8_gsm_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B8_gsm_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B8_gsm_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (GSM)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (GSM)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (GSM)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (GSM)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (GSM)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (GSM)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_gsm_8Hz'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_gsm_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB8_gsm_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E128_gsm_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E128_gsm_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E128_gsm_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B128_gsm_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B128_gsm_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B128_gsm_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (GSM)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (GSM)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (GSM)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (GSM)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (GSM)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (GSM)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_gsm_128Hz'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_gsm_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB128_gsm_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

# FAC座標系を定義、DSI座標系 -> FAC座標系変換行列の作成

# FAC座標系の定義
- z軸は、背景磁場$B_{0}$の単位ベクトルで与える。
- x軸は、反地球方向かつz軸と垂直な単位ベクトルで与える。 ($E_{x}$: Toroidal component, $B_{x}$: Poloidal component)
- y軸は、z軸とx軸の外積で与える。 ($E_{y}$: Poloidal component, $B_{y}$: Toroidal component)

In [ ]:
psp.themis.state(probe='a', trange=time_range, no_update=True)

da_THA_pos_gsm  = pt.data_quants['tha_pos_gsm']
da_THA_pos_gsm  = da_THA_pos_gsm.sortby('time').sel(time=slice(time_range[0], time_range[1]))

da_THA_pos_unit_gsm = da_THA_pos_gsm / np.sqrt((da_THA_pos_gsm * da_THA_pos_gsm).sum(dim='v_dim'))

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
time_width_B_16Hz       = (B16_data_gsm.time[10] - B16_data_gsm.time[9]) / np.timedelta64(1, 's')
da_B_background         = B16_data_gsm.rolling(time=int(background_time_sec/time_width_B_16Hz), center=True).mean('time')
da_B_background_unit    = da_B_background / np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
da_B_background_unit    = da_B_background_unit.dropna(how='all', dim='time')

print(da_B_background_unit)
print(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim')))
print(np.nanmin(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))
print(np.nanmax(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))

In [ ]:
time_array  = da_B_background_unit.time

da_THA_pos_unit_gsm_interp    = da_THA_pos_unit_gsm.interp(time=time_array)

da_u_   = da_THA_pos_unit_gsm_interp - (da_THA_pos_unit_gsm_interp * da_B_background_unit).sum(dim='v_dim') * da_B_background_unit

da_e_z_FAC_inGSM    = da_B_background_unit.drop_attrs()
da_e_x_FAC_inGSM    = (da_u_ / np.sqrt((da_u_ * da_u_).sum(dim='v_dim'))).drop_attrs()
da_e_y_FAC_inGSM    = (xr.apply_ufunc(np.cross, da_e_z_FAC_inGSM, da_e_x_FAC_inGSM, input_core_dims=[['v_dim'], ['v_dim']], output_core_dims=[['v_dim']], vectorize=True)).drop_attrs()

print(da_e_x_FAC_inGSM)
print('')
print(da_e_y_FAC_inGSM)
print('')
print(da_e_z_FAC_inGSM)

In [ ]:
R_FAC_to_GSM = xr.concat(
    [da_e_x_FAC_inGSM, da_e_y_FAC_inGSM, da_e_z_FAC_inGSM],
    dim='axis'
)
R_FAC_to_GSM    = R_FAC_to_GSM.assign_coords(axis=['x_FAC', 'y_FAC', 'z_FAC']).assign_coords(v_dim=np.arange(3))
R_FAC_to_GSM    = R_FAC_to_GSM.dropna(dim='time', how='any')

R_GSM_to_FAC    = R_FAC_to_GSM.transpose('time', 'v_dim', 'axis')

print(R_FAC_to_GSM)
print('')
print(R_GSM_to_FAC)

In [ ]:
da_e_x_GSM_inFAC    = R_GSM_to_FAC.sel(v_dim=0)
da_e_y_GSM_inFAC    = R_GSM_to_FAC.sel(v_dim=1)
da_e_z_GSM_inFAC    = R_GSM_to_FAC.sel(v_dim=2)

In [ ]:
import os
import matplotlib.pyplot as plt

path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330/coordinate_FAC_GSM"
)
os.makedirs(path_base_save_plot, exist_ok=True)

def setup_ax(ax, xlabel, ylabel, title):
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(True, which='both', linestyle=':')
    ax.set_title(title)

def plot_fac_dsi_frame(it, frame_idx, save_dir):
    ex = da_e_x_GSM_inFAC.isel(time=it)
    ey = da_e_y_GSM_inFAC.isel(time=it)
    ez = da_e_z_GSM_inFAC.isel(time=it)

    # FAC 成分
    ex_x = ex.sel(axis='x_FAC').item()
    ex_y = ex.sel(axis='y_FAC').item()
    ex_z = ex.sel(axis='z_FAC').item()

    ey_x = ey.sel(axis='x_FAC').item()
    ey_y = ey.sel(axis='y_FAC').item()
    ey_z = ey.sel(axis='z_FAC').item()

    ez_x = ez.sel(axis='x_FAC').item()
    ez_y = ez.sel(axis='y_FAC').item()
    ez_z = ez.sel(axis='z_FAC').item()

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # ------------- (x, y) plane -------------
    ax = axs[0]
    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
              color='k',   label='FAC-x')
    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
              color='gray', label='FAC-y')
    ax.quiver(0, 0, 0, 0, angles='xy', scale_units='xy', scale=1,
              color='purple', label='FAC-z', linewidth=0)

    ax.quiver(0, 0, ex_x, ex_y, angles='xy', scale_units='xy', scale=1,
              color='r', label='GSM-x')
    ax.quiver(0, 0, ey_x, ey_y, angles='xy', scale_units='xy', scale=1,
              color='b', label='GSM-y')
    ax.quiver(0, 0, ez_x, ez_y, angles='xy', scale_units='xy', scale=1,
              color='g', label='GSM-z')

    setup_ax(ax, 'FAC-x (Radial)', 'FAC-y (Longitudinal)', '(x, y) plane')
    ax.legend(loc='lower left')

    # ------------- (x, z) plane -------------
    ax = axs[1]
    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
              color='k',   label='FAC-x')
    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
              color='purple', label='FAC-z')

    ax.quiver(0, 0, ex_x, ex_z, angles='xy', scale_units='xy', scale=1,
              color='r', label='GSM-x')
    ax.quiver(0, 0, ey_x, ey_z, angles='xy', scale_units='xy', scale=1,
              color='b', label='GSM-y')
    ax.quiver(0, 0, ez_x, ez_z, angles='xy', scale_units='xy', scale=1,
              color='g', label='GSM-z')

    setup_ax(ax, 'FAC-x (Radial)', 'FAC-z (Parallel)', '(x, z) plane')

    # ------------- (y, z) plane -------------
    ax = axs[2]
    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
              color='gray',   label='FAC-y')
    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
              color='purple', label='FAC-z')

    ax.quiver(0, 0, ex_y, ex_z, angles='xy', scale_units='xy', scale=1,
              color='r', label='GSM-x')
    ax.quiver(0, 0, ey_y, ey_z, angles='xy', scale_units='xy', scale=1,
              color='b', label='GSM-y')
    ax.quiver(0, 0, ez_y, ez_z, angles='xy', scale_units='xy', scale=1,
              color='g', label='GSM-z')

    setup_ax(ax, 'FAC-y (Longitudinal)', 'FAC-z (Parallel)', '(y, z) plane')

    fig.suptitle(str(da_e_x_GSM_inFAC.time.values[it]), fontsize=14)
    plt.tight_layout()

    # ファイル名：time index をゼロ埋め
    fname = os.path.join(save_dir, f"coord_{frame_idx:06d}.png")
    fig.savefig(fname, dpi=150)
    plt.close(fig)


In [ ]:
#from concurrent.futures import ProcessPoolExecutor, as_completed
#import multiprocessing as mp
#from tqdm import tqdm
#import numpy as np
#
## --- 1) タスク作成 ---
#n_time = da_e_x_GSM_inFAC.sizes['time']
#step = 16 * 60   # 64Hz × 60 sec
#
#tasks = []
#frame_idx = 0
#for it in range(0, n_time, step):
#    tasks.append((it, frame_idx))
#    frame_idx += 1
#
#print("num frames:", len(tasks))
#
#
## --- 2) worker ---
#def worker(args):
#    it, frame_idx, save_path = args
#    plot_fac_dsi_frame(it, frame_idx, save_path)
#    return frame_idx
#
#
## --- 3) 並列 + tqdm ---
#save_path = path_base_save_plot
#n_workers = max(1, mp.cpu_count() - 1)
#
#with ProcessPoolExecutor(max_workers=n_workers) as exe:
#    futures = [
#        exe.submit(worker, (it, idx, save_path))
#        for it, idx in tasks
#    ]
#
#    for f in tqdm(as_completed(futures), total=len(futures)):
#        _ = f.result()   # 例外を拾うため

# GSM座標系 -> FAC座標系変換の実行

In [ ]:
ds_EBspin_fac_segs  = []

for ds_seg in ds_EBspin_gsm_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_GSM_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['Espin_gsm_x'], ds_seg['Espin_gsm_y'], ds_seg['Espin_gsm_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['Bspin_gsm_x'], ds_seg['Bspin_gsm_y'], ds_seg['Bspin_gsm_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'Espin_fac_x',
            'y_FAC': 'Espin_fac_y',
            'z_FAC': 'Espin_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'Bspin_fac_x',
            'y_FAC': 'Bspin_fac_y',
            'z_FAC': 'Bspin_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds]).dropna(dim='time', how='any')
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EBspin_fac_segs.append(ds_fac)

In [ ]:
ds_EB8_fac_segs = []

for ds_seg in ds_EB8_gsm_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_GSM_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['E8_gsm_x'], ds_seg['E8_gsm_y'], ds_seg['E8_gsm_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['B8_gsm_x'], ds_seg['B8_gsm_y'], ds_seg['B8_gsm_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'E8_fac_x',
            'y_FAC': 'E8_fac_y',
            'z_FAC': 'E8_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'B8_fac_x',
            'y_FAC': 'B8_fac_y',
            'z_FAC': 'B8_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds]).dropna(dim='time', how='any')
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EB8_fac_segs.append(ds_fac)


In [ ]:
ds_EB128_fac_segs = []

for ds_seg in ds_EB128_gsm_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_GSM_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['E128_gsm_x'], ds_seg['E128_gsm_y'], ds_seg['E128_gsm_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['B128_gsm_x'], ds_seg['B128_gsm_y'], ds_seg['B128_gsm_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'E128_fac_x',
            'y_FAC': 'E128_fac_y',
            'z_FAC': 'E128_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'B128_fac_x',
            'y_FAC': 'B128_fac_y',
            'z_FAC': 'B128_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds]).dropna(dim='time', how='any')
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EB128_fac_segs.append(ds_fac)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330"
)
os.makedirs(path_base_save_plot, exist_ok=True)

mpl.rcParams['font.size'] = 15

# ---- 5分刻みの時間窓 ----
t_start = np.datetime64('2022-09-01T22:25:00')
t_end   = np.datetime64('2022-09-01T23:10:00')
step_min    = 45
step    = np.timedelta64(step_min, 'm')

t_list = []
t0 = t_start
while t0 < t_end:
    t1 = t0 + step
    t_list.append((t0, t1))
    t0 = t1

def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
    """全 seg を重ねて [t0, t1] の波形を描いて保存"""

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(6, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)
    ax_4.tick_params(axis='x', which='both', labelbottom=False)

    # ---- 各 seg を同じ軸に重ね描き ----
    for ds_seg in ds_list:
        ds_win = ds_seg.sel(time=slice(t0, t1))
        if ds_win.time.size == 0:
            continue

        ax_0.plot(ds_win.time, ds_win['E128_fac_x'], lw=1, c='k')
        ax_1.plot(ds_win.time, ds_win['E128_fac_y'], lw=1, c='k')
        ax_2.plot(ds_win.time, ds_win['E128_fac_z'], lw=1, c='k')
        ax_3.plot(ds_win.time, ds_win['B128_fac_x'], lw=1, c='k')
        ax_4.plot(ds_win.time, ds_win['B128_fac_y'], lw=1, c='k')
        ax_5.plot(ds_win.time, ds_win['B128_fac_z'], lw=1, c='k')

    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')

    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
        ax.minorticks_on()
        ax.grid(which='both', alpha=0.5)

    ax_5.set_xlim(t0, t1)
    fig.tight_layout()

    if os.path.isdir(save_dir):
        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac_128Hz'
        os.makedirs(save_dir_, exist_ok=True)
        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
        fpath = os.path.join(save_dir_, fname)
        fig.savefig(fpath)
        plt.close(fig)
    else:
        plt.show()
        plt.close()


from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

def worker(args):
    """並列実行するためのラッパー"""
    ds_list, t0, t1, save_dir = args
    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
    return str(t0)  # ログ用

# 並列実行用のタスクをまとめる
tasks = [(ds_EB128_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]

# CPUコア数
n_cores = max(1, mp.cpu_count())

with ProcessPoolExecutor(max_workers=n_cores) as exe:
    for out in exe.map(worker, tasks):
        print("done:", out)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['Espin_fac_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['Espin_fac_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['Espin_fac_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['Bspin_fac_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['Bspin_fac_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['Bspin_fac_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac_spin'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EBspin_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E8_fac_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E8_fac_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E8_fac_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B8_fac_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B8_fac_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B8_fac_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac_8Hz'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB8_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E128_fac_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E128_fac_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E128_fac_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B128_fac_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B128_fac_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B128_fac_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac_128Hz'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB128_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

# 軌道データから、衛星速度(GSM)を導出

In [ ]:
v_sc_gsm    = pt.data_quants['tha_vel_gsm'].sortby('time')

In [ ]:
path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330"
)

In [ ]:
#import matplotlib.pyplot as plt
#import datetime
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_gsm_analysis  = v_sc_gsm.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_gsm_analysis.time, v_sc_gsm_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_sc_gsm_analysis.time, v_sc_gsm_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_sc_gsm_analysis.time, v_sc_gsm_analysis.data[:, 2], lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (GSM)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (GSM)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (GSM)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_sc_gsm_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_gsm.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
R_interp    = R_GSM_to_FAC.interp(time=v_sc_gsm.time)

v_sc_fac    = xr.dot(v_sc_gsm, R_interp, dims='v_dim')
v_sc_fac    = v_sc_fac.dropna(dim='time', how='any')
print(v_sc_fac)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_fac_analysis  = v_sc_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 2], lw=1, c='k')
#ax_3.plot(v_sc_fac_analysis.time, np.sqrt(v_sc_fac_analysis.data[:, 0]**2E0 + v_sc_fac_analysis.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sc}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sc_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# イオン流速($\approx$ MHD流速)、イオン温度$\rightarrow$イオン熱速度、電子温度$\rightarrow$ ion acoustic sppedの導出

In [ ]:
import pyspedas as psp
import pytplot as pt

psp.themis.mom(trange=time_range, probe='a', level='l2', no_update=True)

In [ ]:
ND_electron     = pt.data_quants['tha_peem_density'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all') # [/cc]
Temp_electron   = pt.data_quants['tha_peem_ptot'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all') # [eV]

ND_ion          = pt.data_quants['tha_peim_density'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all') # [/cc]
Temp_ion        = pt.data_quants['tha_peim_ptot'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all') # [eV]
v_ion_gsm       = pt.data_quants['tha_peim_velocity_gsm'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')    # [km/s]

In [ ]:
R_interp    = R_GSM_to_FAC.interp(time=v_ion_gsm.time)

v_ion_fac   = xr.dot(v_ion_gsm, R_interp, dims='v_dim')
v_ion_fac   = v_ion_fac.dropna(dim='time', how='any')

print(v_ion_fac)
print(v_sc_fac)

In [ ]:
v_sys_fac   = v_ion_fac - v_sc_fac.interp(time=v_ion_fac.time, method='linear')
v_sys_fac   = v_sys_fac.dropna(dim='time', how='any')
print(v_sys_fac)
print(v_sys_fac.time)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_ion_gsm_analysis  = v_ion_gsm.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_gsm_analysis.time, v_ion_gsm_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_ion_gsm_analysis.time, v_ion_gsm_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_ion_gsm_analysis.time, v_ion_gsm_analysis.data[:, 2], lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (GSM)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (GSM)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (GSM)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_ion_gsm_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_gsm.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_ion_fac_analysis  = v_ion_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 2], lw=1, c='k')
#ax_3.plot(v_ion_fac_analysis.time, np.sqrt(v_ion_fac_analysis.data[:, 0]**2E0 + v_ion_fac_analysis.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{ion}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_ion_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt

time_range_analysis     = ['20220901/23:05:45', '20220901/23:08:15']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

dt_v_ion_fac        = (v_ion_fac.time.data[1] - v_ion_fac.time.data[0]) / np.timedelta64(1, 's')
v_ion_fac_mean      = v_ion_fac.rolling(time=int(100/dt_v_ion_fac), center=True).mean()
v_ion_fac_analysis  = v_ion_fac_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(4, 1)
ax_0 = fig.add_subplot(gs[0, 0])
ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 0], lw=1, c='k')
ax_1.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 1], lw=1, c='k')
ax_2.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 2], lw=1, c='k')
ax_3.plot(v_ion_fac_analysis.time, np.sqrt(v_ion_fac_analysis.data[:, 0]**2E0 + v_ion_fac_analysis.data[:, 1]**2E0), lw=1, c='k')

ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (FAC)' + '\n' + '[km/s]')
ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (FAC)' + '\n' + '[km/s]')
ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (FAC)' + '\n' + '[km/s]')
ax_3.set_ylabel(r'$V_{\mathrm{ion}\perp}$ (FAC)' + '\n' + '[km/s]')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.5)
ax_3.minorticks_on()
ax_3.grid(which='both', alpha=0.5)

ax_3.set_xlim(v_ion_fac_analysis.time.values[[0, -1]])

fig.tight_layout()

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'v_ion_fac_mean_event3.png')
    print(fig_path)
    fig.savefig(fig_path)
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sys_fac_analysis  = v_sys_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 2], lw=1, c='k')
#ax_3.plot(v_sys_fac_analysis.time, np.sqrt(v_sys_fac_analysis.data[:, 0]**2E0 + v_sys_fac_analysis.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sys_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#import matplotlib as mpl
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 20
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_v_sys_fac        = (v_sys_fac.time.data[1] - v_sys_fac.time.data[0]) / np.timedelta64(1, 's')
#v_sys_fac_mean      = v_sys_fac.rolling(time=int(100/dt_v_sys_fac), center=True).mean()
#v_sys_fac_analysis_mean     = v_sys_fac_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 0], lw=1, c='k')
#ax_1.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 1], lw=1, c='k')
#ax_2.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 2], lw=1, c='k')
#ax_3.plot(v_sys_fac_analysis_mean.time, np.sqrt(v_sys_fac_analysis_mean.data[:, 0]**2E0 + v_sys_fac_analysis_mean.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sys_fac_analysis_mean.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac_mean.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

- Alfvén speed
```math
v_{\mathrm{A}} := \frac{B_{0}}{\sqrt{\mu_{0} n_{\mathrm{e}} m_{\mathrm{i}}}}
```
- Ion thermal speed
```math
v_{\mathrm{thi}} := \sqrt{\frac{2 T_{\mathrm{i}}}{m_{\mathrm{i}}}}
```
- Ion acoustic speed
```math
c_{\mathrm{s}} := \sqrt{\frac{T_{\mathrm{e}}}{m_{\mathrm{i}}}}
```
- Proton cyclotron frequency
```math
f_{\mathrm{p}} := \frac{1}{2 \pi} \frac{e B_{0}}{m_{\mathrm{p}}}
```
- Ion plasma beta
```math
\beta_{\mathrm{i}} := \frac{2 \mu_{0} n_{\mathrm{e}} T_{\mathrm{i}}}{B_{0}^{2}} = \left( \frac{v_{\mathrm{thi}}}{v_{\mathrm{A}}} \right)^{2}
```
- Ion-to-electron temperature ratio
```math
\tau := \frac{T_{\mathrm{i}}}{T_{\mathrm{e}}} = \frac{1}{2} \left( \frac{v_{\mathrm{thi}}}{c_{\mathrm{s}}} \right)^{2}
```

In [ ]:
B_total = pt.data_quants['tha_fgs_btotal'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')   # [nT]

In [ ]:
time_base   = B_total.time
print(time_base)

ND_electron_interp  = ND_electron.interp(time=time_base, method='linear')

Temp_electron_interp    = Temp_electron.interp(time=time_base, method='linear')
Temp_ion_interp         = Temp_ion.interp(time=time_base, method='linear')
v_sys_fac_interp        = v_sys_fac.interp(time=time_base, method='linear') * 1E3   # [m/s]
v_sys_fac_perp_interp   = xr.DataArray(
    data=np.sqrt(v_sys_fac_interp.data[:, 0]**2E0 + v_sys_fac_interp.data[:, 1]**2E0),
    dims=['time'],
    coords={'time': v_sys_fac_interp.time},
    attrs=v_sys_fac_interp.attrs
)

proton_mass = 1.6726219e-27  # kg
elementary_charge = 1.60218e-19  # C
mu0 = 4*np.pi*1e-7

Alfven_speed        = B_total*1E-9 / np.sqrt(mu0 * ND_electron_interp*1E6 * proton_mass)
ion_thermal_speed   = np.sqrt(2E0 * Temp_ion_interp*elementary_charge / proton_mass)
ion_acoustic_speed  = np.sqrt(Temp_electron_interp*elementary_charge / proton_mass)

electron_mass_kg        = 9.1093837E-31
electron_thermal_speed  = np.sqrt(2E0 * Temp_electron_interp*elementary_charge / electron_mass_kg)

proton_cycl_freq    = elementary_charge * B_total*1E-9 / proton_mass / 2E0 / np.pi

ion_plasma_beta     = (ion_thermal_speed / Alfven_speed)**2E0
ion_to_electron_temp_ratio  = (ion_thermal_speed / ion_acoustic_speed)**2E0 / 2E0

# moving mean
dt_time_base            = (time_base.data[10] - time_base.data[9]) / np.timedelta64(1, 's')
print(dt_time_base)

Alfven_speed_mean       = Alfven_speed.rolling(time=int(100/dt_time_base), center=True).mean()
ion_thermal_speed_mean  = ion_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
electron_thermal_speed_mean = electron_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
ion_acoustic_speed_mean = ion_acoustic_speed.rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_perp_mean     = v_sys_fac_perp_interp.rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_x_mean        = v_sys_fac_interp[:, 0].rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_y_mean        = v_sys_fac_interp[:, 1].rolling(time=int(100/dt_time_base), center=True).mean()

ion_plasma_beta_mean            = ion_plasma_beta.rolling(time=int(100/dt_time_base), center=True).mean()
ion_to_electron_temp_ratio_mean = ion_to_electron_temp_ratio.rolling(time=int(100/dt_time_base), center=True).mean()
proton_cycl_freq_mean           = proton_cycl_freq.rolling(time=int(100/dt_time_base), center=True).mean()

# DataSet格納
ds_velocity_ms_perp = xr.Dataset(
    {
        'Alfven_speed':             Alfven_speed_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_sys_speed':           v_sys_fac_perp_mean
    }
)
ds_velocity_ms_perp = ds_velocity_ms_perp

print(ds_velocity_ms_perp)

ds_velocity_ms_toroidal = xr.Dataset(
    {
        'Alfven_speed':             Alfven_speed_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_sys_speed':           np.abs(v_sys_fac_x_mean)
    }
)
ds_velocity_ms_toroidal = ds_velocity_ms_toroidal

print(ds_velocity_ms_toroidal)

ds_velocity_ms_poloidal = xr.Dataset(
    {
        'Alfven_speed':             Alfven_speed_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_sys_speed':           np.abs(v_sys_fac_y_mean)
    }
)
ds_velocity_ms_poloidal = ds_velocity_ms_poloidal

print(ds_velocity_ms_poloidal)

ds_parameter = xr.Dataset(
    {
        'ion_plasma_beta':      ion_plasma_beta_mean,
        'i-e_temp_ratio':       ion_to_electron_temp_ratio_mean,
        'proton_cycl_freq_Hz':  proton_cycl_freq_mean,
        'number_density_cc':    ND_electron_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_ion_eV':          Temp_ion_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_electron_eV':     Temp_electron_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'B_total_nT':           B_total.rolling(time=int(100/dt_time_base), center=True).mean()
    }
)
ds_parameter = ds_parameter

print(ds_parameter)

In [ ]:
def dedup_and_sort(ds):
    ds = ds.sortby("time")
    t = ds["time"].values
    _, keep = np.unique(t, return_index=True)  # 先勝ちで一意化
    return ds.isel(time=np.sort(keep))

ds_parameter            = dedup_and_sort(ds_parameter)
ds_velocity_ms_perp     = dedup_and_sort(ds_velocity_ms_perp)
ds_velocity_ms_poloidal = dedup_and_sort(ds_velocity_ms_poloidal)
ds_velocity_ms_toroidal = dedup_and_sort(ds_velocity_ms_toroidal)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed']*1E-3,       lw=1, c='k')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3,  lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3, lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,     lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_perp.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed']*1E-3,       lw=1, c='k')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3,  lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3, lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,     lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_poloidal.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed']*1E-3,       lw=1, c='k')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3,  lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3, lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,     lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_toroidal.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 15))
#gs = fig.add_gridspec(7, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#ax_5.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_cc'],   lw=1, c='k')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],         lw=1, c='k')
#ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],    lw=1, c='k')
#ax_3.plot(ds_parameter_analysis.time, ds_parameter_analysis['i-e_temp_ratio'],      lw=1, c='k')
#ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta'],     lw=1, c='k')
#ax_5.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],          lw=1, c='k')
#ax_6.plot(ds_parameter_analysis.time, ds_parameter_analysis['proton_cycl_freq_Hz'], lw=1, c='k')
#
#ax_0.set_ylabel(r'$n_{\mathrm{e}}$'     + '\n' + '[/cc]')
#ax_1.set_ylabel(r'$T_{\mathrm{i}}$'     + '\n' + '[eV]')
#ax_2.set_ylabel(r'$T_{\mathrm{e}}$'     + '\n' + '[eV]')
#ax_3.set_ylabel(r'$\tau$')
#ax_4.set_ylabel(r'$\beta_{\mathrm{i}}$')
#ax_5.set_ylabel(r'$B_{0}$'              + '\n' + '[nT]')
#ax_6.set_ylabel(r'$f_{\mathrm{H}^{+}}$' + '\n' + '[Hz]')
#
#ax_3.set_yscale('log')
#ax_4.set_yscale('log')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#ax_6.minorticks_on()
#ax_6.grid(which='both', alpha=0.5)
#
#ax_6.set_xlim(ds_parameter_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'parameter_summary.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
psp.themis.state(trange=time_range, probe='a', no_update=True)
psp.cotrans(name_in='tha_pos_gsm', name_out='tha_pos_sm', coord_in='gsm', coord_out='sm')
THA_SM_pos = pt.data_quants['tha_pos_sm'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
print(THA_SM_pos)

THA_rmlatmlt_R = np.sqrt(THA_SM_pos.data[:, 0]**2E0 + THA_SM_pos.data[:, 1]**2E0 + THA_SM_pos.data[:, 2]**2E0) / 6378.1
THA_rmlatmlt_MLAT = np.rad2deg(np.arctan2(THA_SM_pos.data[:, 2], np.sqrt(THA_SM_pos.data[:, 0]**2E0 + THA_SM_pos.data[:, 1]**2E0)))
THA_rmlatmlt_MLT = np.rad2deg(np.arctan2(THA_SM_pos.data[:, 1], THA_SM_pos.data[:, 0])) / 15. + 12.

print(THA_rmlatmlt_R)
print(THA_rmlatmlt_MLAT)
print(THA_rmlatmlt_MLT)

```math
\theta_{\mathrm{GSM}} := \mathrm{arctan} \left( \frac{B_{\mathrm{GSM}z}}{\sqrt{B_{\mathrm{GSM}x}^{2} + B_{\mathrm{GSM}y}^{2}}} \right)
```
[Lui et al., 1999; Duan et al., 2011]

In [ ]:
theta_GSM = np.rad2deg(np.arctan(B16_data_gsm.data[:, 2] / np.sqrt(B16_data_gsm.data[:, 0]**2E0 + B16_data_gsm.data[:, 1]**2E0)))

da_theta_GSM = xr.DataArray(
    data=theta_GSM,
    dims=['time'],
    coords={'time': B16_data_gsm.time},
    name='theta_GSM_deg'
)

da_theta_dt  = (da_theta_GSM.time[1] - da_theta_GSM.time[0]) / np.timedelta64(1, 's')
da_theta_GSM = da_theta_GSM.rolling(time=int(100/da_theta_dt), center=True).mean()

da_theta_GSM

In [ ]:
da_B16_data_gsm_z = xr.DataArray(
    data=B16_data_gsm.data[:, 2],
    dims=['time'],
    coords={'time': B16_data_gsm.time},
    name='B16_gsm_z'
)

da_B16_data_gsm_z = da_B16_data_gsm_z.rolling(time=int(100/da_theta_dt), center=True).mean()

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#da_theta_GSM_analysis   = da_theta_GSM.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#import matplotlib.ticker as mticker
#from datetime import datetime
#import matplotlib.dates as mdates
#
#mpl.rcParams['font.size'] = 25
#
#fig = plt.figure(figsize=(11, 21))
#gs = fig.add_gridspec(8, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_2_share = ax_2.twinx()
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
#ax_7 = fig.add_subplot(gs[7, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#ax_5.tick_params(axis='x', which='both', labelbottom=False)
#ax_6.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_cc'],           lw=1, c='k')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],                 lw=1, c='red', label=r'$T_{\mathrm{i}}$')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],            lw=1, c='blue', label=r'$T_{\mathrm{e}}$')
#ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['proton_cycl_freq_Hz'],         lw=1, c='k')
#ax_2_share.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],            lw=1, c='k')
#ax_3.plot(da_theta_GSM_analysis.time, da_theta_GSM_analysis.data,                           lw=1, c='k')
#ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta'],             lw=1, c='k')
#ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=2, c='b', linestyle='-.')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed']*1E-3,       lw=1, c='k', label=r'$v_{\mathrm{A}}$')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=2, c='red', linestyle='-.', label=r'$v_{\mathrm{thi}}$')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3, lw=1, c='magenta', label=r'$c_{\mathrm{s}}$')
#ax_7.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,     lw=1, c='k')
#
#ax_0.set_ylabel(r'$n_{\mathrm{e}}$'                     + '\n' + '[/cc]')
#ax_1.set_ylabel(r'$T_{\mathrm{i}}$, $T_{\mathrm{e}}$'   + '\n' + '[eV]')
#ax_2.set_ylabel(r'$f_{\mathrm{H}^{+}}$'                 + '\n' + '[Hz]')
#ax_2_share.set_ylabel(r'$B_{0}$'                        + '\n' + '[nT]')
#ax_3.set_ylabel(r'$\theta_{\mathrm{GSM}}$'              + '\n' + r'[deg]')
#ax_4.set_ylabel(r'$\beta_{\mathrm{i}}$')
#ax_5.set_ylabel(r'$v_{\mathrm{the}}$'                   + '\n' + '[km/s]')
#ax_6.set_ylabel(r'$v_{\mathrm{A}}$, $v_{\mathrm{thi}}$, $c_{\mathrm{s}}$'   + '\n' + '[km/s]')
#ax_7.set_ylabel(r'$V_{\mathrm{sys}\perp}$'              + '\n' + '[km/s]')
#
##ax_0.set_yscale('log')
##ax_1.set_yscale('log')
#ax_1.set_ylim(ymin=1)
#ax_4.set_yscale('log')
##ax_5.set_yscale('log')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#ax_6.minorticks_on()
#ax_6.grid(which='both', alpha=0.5)
#ax_7.minorticks_on()
#ax_7.grid(which='both', alpha=0.5)
#
#ax_1.legend(fontsize=13, ncol=2)
#ax_6.legend(fontsize=13, ncol=3)
#
#ax_7.set_xlim(ds_parameter_analysis.time.values[[0, -1]])
#
#def add_panel_label(ax, label, x=-0.15, y=0.95):
#    ax.text(x, y, label, transform=ax.transAxes,
#            ha='right', va='bottom', clip_on=False)
#
#def to_py_datetime(t_np64):
#    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)
#
## 軌道データ
#t_pos_py = to_py_datetime(THA_SM_pos.time.values)
#t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数
#
#R   = np.asarray(THA_rmlatmlt_R, dtype=float)  # Re
#mlat= np.asarray(THA_rmlatmlt_MLAT, dtype=float)  # deg
#mlt = np.asarray(THA_rmlatmlt_MLT, dtype=float)  # hour [0,24)
#
## --- MLT の 24h 周期をほどいてから補間し、最後に 24 で折り返す ---
#mlt_unwrap = np.unwrap(mlt * 2*np.pi/24.0) * 24.0/(2*np.pi)
#
## 補間関数（tick の x は「日数」なのでそのまま使う）
#def interp_at(x_num):
#    Ri    = np.interp(x_num, t_pos_num, R, left=np.nan, right=np.nan)
#    mlati = np.interp(x_num, t_pos_num, mlat, left=np.nan, right=np.nan)
#    mltiu = np.interp(x_num, t_pos_num, mlt_unwrap, left=np.nan, right=np.nan)
#    mlti  = np.mod(mltiu, 24.0)
#    return Ri, mlati, mlti
#
## 目盛フォーマッタ
#def rmlt_formatter(x, pos=None):
#    Ri, mlati, mlti = interp_at(x)
#    if np.any(~np.isfinite([Ri, mlati, mlti])):
#        return ""  # 範囲外は空
#    return (f"{Ri:0.2f}\n"
#            f"{mlati:0.2f}\n"
#            f"{mlti:0.2f}")
#
## セカンダリ x 軸（底 side）を作ってラベルを差し替え
#secax = ax_7.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
#secax.xaxis.set_major_formatter(mticker.FuncFormatter(rmlt_formatter))
#
## メインの時間ラベルと重ならないよう余白を広げる
#ax_7.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
#secax.tick_params(axis='x', which='major', pad=60)  # R/MLAT/MLTラベル
#
## 好みで：目盛間隔をメイン x と合わせる
#secax.set_ticks(ax_7.get_xticks())
#
#fig.text(0.05, 0.085, "hhmm", ha='center', va='center')
#fig.text(0.05, 0.048, r"R [$R_{\mathrm{E}}$]", ha='center', va='center')
#fig.text(0.05, 0.028, r"MLAT", ha='center', va='center')
#fig.text(0.05, 0.008, r"MLT", ha='center', va='center')
#
#add_panel_label(ax_0, '(b-1)')
#add_panel_label(ax_1, '(b-2)')
#add_panel_label(ax_2, '(b-3)')
#add_panel_label(ax_3, '(b-4)')
#add_panel_label(ax_4, '(b-5)')
#add_panel_label(ax_5, '(b-6)')
#add_panel_label(ax_6, '(b-7)')
#add_panel_label(ax_7, '(b-8)')
#
#fig.suptitle('THEMIS-A', y=0.99)
#
#fig.subplots_adjust(hspace=0)
#fig.tight_layout(pad=0)
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'Figure_2b.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# KAWの確認に適した時間窓$T_{\mathrm{window}}$の検討

```math
\frac{1}{v_{\mathrm{A}}} \frac{|\bf{E}_{\perp}|}{|\bf{B}_{\perp}|} = \frac{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2}}{\sqrt{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2} \left( 1 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)}} = \sqrt{10} \\
k_{\perp} \rho_{\mathrm{i}} = \frac{f_{\mathrm{sc}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \\
\therefore T_{\mathrm{window}} := \frac{1}{f_{\mathrm{sc}}} = \frac{1}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \left[ 9 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \left\{ 10 + \sqrt{117 \left( \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)^{2} + 180 \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} + 100} \right\} \right]^{-\frac{1}{2}}
```

In [ ]:
T_window = 1. / ds_parameter['proton_cycl_freq_Hz'].data * ds_velocity_ms_perp['ion_thermal_speed'].data / ds_velocity_ms_perp['perp_sys_speed'].data / np.sqrt(9. + 1. / ds_parameter['i-e_temp_ratio'].data * (10. + np.sqrt(117. / (ds_parameter['i-e_temp_ratio'].data)**(2.) + 180. / ds_parameter['i-e_temp_ratio'].data + 100.)))

da_T_window = xr.DataArray(data=T_window, dims=('time'), coords={'time': ds_velocity_ms_perp.time}, name='T_window')
da_T_window

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_T_window_analysis = da_T_window.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

print(np.nanmin(da_T_window_analysis))

In [ ]:
import matplotlib as mpl

time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_T_window_analysis = da_T_window.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

mpl.rcParams['font.size'] = 15
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(111)
ax.plot(da_T_window_analysis.time, da_T_window_analysis.data, c='k', lw=1)
ax.minorticks_on()
ax.set_ylabel(r'$T_{\mathrm{window}}$' + '\n[sec]')
ax.set_yscale('log')
ax.set_ylim(ymin=1)
ax.grid(which='both', alpha=0.5)
ax.set_xlim(da_T_window_analysis.time.values[[0, -1]])

fig.tight_layout()

print(np.nanmin(da_T_window_analysis), np.nanmean(da_T_window_analysis))

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'T_window.png')
    print(fig_path)
    fig.savefig(fig_path)
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

# Wavelet analysis

In [ ]:
import os
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt
import pywt

import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.tdwavelet_themis as tw
import importlib
importlib.reload(tw)

In [ ]:
fs = 0.3648
dt = 1.0 / fs
s0 = 2.0
dj = 1.0/32.0

f_target = 0.01         # 目標最低周波数
N_cycle_min = 3.0       # 最低3周期は欲しい

# ---- 1) 全セグメントの長さを調べる ----
T_list = []
for ds_seg in ds_EB8_fac_segs:
    t0 = ds_seg.time.values[0]
    t1 = ds_seg.time.values[-1]
    T_seg = (t1 - t0) / np.timedelta64(1, 's')
    T_list.append(T_seg)

T_max = max(T_list)
print("max segment length [s] =", T_max)

# ---- 2) 最長セグメントで意味のある最低周波数 ----
f_min_phys = N_cycle_min / T_max
f_min_seg = f_min_phys #max(f_target, f_min_phys)
print("f_min_seg (for longest segment) =", f_min_seg)

# ---- 3) その周波数に対応する scale_max ----
scale_target = 1.0 / (f_min_seg * dt)

# ---- 4) J_longest を計算 ----
J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
print("J_longest =", J_longest)

ds_EBspin_fac_cwt_segs    = []
EBspin_fac_vars = ['Espin_fac_x', 'Espin_fac_y', 'Espin_fac_z', 'Bspin_fac_x', 'Bspin_fac_y', 'Bspin_fac_z']

for ds_seg in ds_EBspin_fac_segs:
    ds_cwt = tw.cwt_from_dataset(
        ds_seg,
        dt=dt,
        s0=s0,
        dj=dj,
        J=J_longest,     # << 全てこれを使う
        variables=EBspin_fac_vars
    )
    ds_EBspin_fac_cwt_segs.append(ds_cwt)

print(ds_EBspin_fac_cwt_segs)

In [ ]:
fs = 8.0
dt = 1.0 / fs
s0 = 2.0
dj = 1.0/32.0

f_target = 0.01         # 目標最低周波数
N_cycle_min = 3.0       # 最低3周期は欲しい

# ---- 1) 全セグメントの長さを調べる ----
T_list = []
for ds_seg in ds_EB8_fac_segs:
    t0 = ds_seg.time.values[0]
    t1 = ds_seg.time.values[-1]
    T_seg = (t1 - t0) / np.timedelta64(1, 's')
    T_list.append(T_seg)

T_max = max(T_list)
print("max segment length [s] =", T_max)

# ---- 2) 最長セグメントで意味のある最低周波数 ----
f_min_phys = N_cycle_min / T_max
f_min_seg = f_min_phys #max(f_target, f_min_phys)
print("f_min_seg (for longest segment) =", f_min_seg)

# ---- 3) その周波数に対応する scale_max ----
scale_target = 1.0 / (f_min_seg * dt)

# ---- 4) J_longest を計算 ----
J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
print("J_longest =", J_longest)

ds_EB8_fac_cwt_segs    = []
EB8_fac_vars = ['E8_fac_x', 'E8_fac_y', 'E8_fac_z', 'B8_fac_x', 'B8_fac_y', 'B8_fac_z']

for ds_seg in ds_EB8_fac_segs:
    ds_cwt = tw.cwt_from_dataset(
        ds_seg,
        dt=dt,
        s0=s0,
        dj=dj,
        J=J_longest,     # << 全てこれを使う
        variables=EB8_fac_vars
    )
    ds_EB8_fac_cwt_segs.append(ds_cwt)

print(ds_EB8_fac_cwt_segs)

In [ ]:
fs = 128.0
dt = 1.0 / fs
s0 = 2.0
dj = 1.0/32.0

f_target = 4.0          # 目標最低周波数
N_cycle_min = 3.0       # 最低3周期は欲しい

# ---- 1) 全セグメントの長さを調べる ----
T_list = []
for ds_seg in ds_EB128_fac_segs:
    t0 = ds_seg.time.values[0]
    t1 = ds_seg.time.values[-1]
    T_seg = (t1 - t0) / np.timedelta64(1, 's')
    T_list.append(T_seg)

T_max = max(T_list)
print("max segment length [s] =", T_max)

# ---- 2) 最長セグメントで意味のある最低周波数 ----
f_min_phys = N_cycle_min / T_max
f_min_seg = f_min_phys #max(f_target, f_min_phys)
print("f_min_seg (for longest segment) =", f_min_seg)

# ---- 3) その周波数に対応する scale_max ----
scale_target = 1.0 / (f_min_seg * dt)

# ---- 4) J_longest を計算 ----
J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
print("J_longest =", J_longest)

ds_EB128_fac_cwt_segs    = []
EB128_fac_vars = ['E128_fac_x', 'E128_fac_y', 'E128_fac_z', 'B128_fac_x', 'B128_fac_y', 'B128_fac_z']

for ds_seg in ds_EB128_fac_segs:
    ds_cwt = tw.cwt_from_dataset(
        ds_seg,
        dt=dt,
        s0=s0,
        dj=dj,
        J=J_longest,     # << 全てこれを使う
        variables=EB128_fac_vars
    )
    ds_EB128_fac_cwt_segs.append(ds_cwt)

print(ds_EB128_fac_cwt_segs)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

# ---- セグメント連結（freq合わせ）----
def concat_cwt_segments(dsets, var):
    # dsets をリストに正規化
    if isinstance(dsets, xr.Dataset):
        dsets = [dsets]
    elif isinstance(dsets, (str, bytes)):
        raise TypeError("dsets は Dataset のリストにして")

    das = []
    for ds in dsets:
        if ds is None or not isinstance(ds, xr.Dataset):
            continue
        if var in ds.data_vars:
            das.append(ds[var])

    if not das:
        return None, None

    pow_cat = xr.concat(das, dim="time").sortby("time")

    coi_name = var.replace("_cwt", "_coi")
    coi_list = []
    for ds in dsets:
        if isinstance(ds, xr.Dataset) and coi_name in ds.data_vars:
            coi_list.append(ds[coi_name])
    coi_cat = xr.concat(coi_list, dim="time").sortby("time") if coi_list else None
    return pow_cat, coi_cat

# ---- 1面描画：外でax/caxを用意する ----
def plot_cwt_on_ax(ax, da_pow, da_coi=None, t0=None, minutes=5,
                   zrange=(1e-6, 1e3), yrange=(1e-2, 4.0),
                   cmap="turbo", label_left="", unit_right=""):
    # 時間切り出し
    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        da = da_pow.sel(time=slice(t0, t1))
        coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None else None
        #ax.set_xlim(t0, t1)
    else:
        da, coi = da_pow, da_coi
    if da.time.size == 0: return None, None

    T = mdates.date2num(da.time.values)
    F = da.freq.values
    Z = da.values.astype(float)

    # COIマスク（低周波側をNaN）
    if coi is not None:
        C = coi.values[:, None]
        Z = np.where(F[None, :] < C, np.nan, Z)

    # メッシュ
    Tm = np.tile(T, (F.size, 1)).T
    Fm = np.tile(F, (T.size, 1))

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto",
                        norm=LogNorm(vmin=zrange[0], vmax=zrange[1]), cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)


    # 右側カラーバー
    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(f"{unit_right}")
    return pcm, cb


In [ ]:
targets_EB128 = [
    ("E128_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E128_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E128_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B128_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B128_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B128_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]
joined_EB128_fac_cwt = {}
for v, _, _ in targets_EB128:
    da, coi = concat_cwt_segments(ds_EB128_fac_cwt_segs, v)
    if da is not None: joined_EB128_fac_cwt[v] = (da, coi)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n
#    for n in range(9)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB128), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets_EB128):
#        if v not in joined_EB128_fac_cwt: continue
#        da, coi = joined_EB128_fac_cwt[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(np.nanmin(da.freq), np.nanmax(da.freq)),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_high_freq_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
targets_EB8 = [
    ("E8_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E8_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E8_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B8_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B8_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B8_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]
joined_EB8_fac_cwt = {}
for v, _, _ in targets_EB8:
    da, coi = concat_cwt_segments(ds_EB8_fac_cwt_segs, v)
    if da is not None: joined_EB8_fac_cwt[v] = (da, coi)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T20:00') + np.timedelta64(5, 'm')*n
#    for n in range(48)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB8), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets_EB8):
#        if v not in joined_EB8_fac_cwt: continue
#        da, coi = joined_EB8_fac_cwt[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(np.nanmin(da.freq), np.nanmax(da.freq)),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_low_freq_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
targets_EBspin = [
    ("Espin_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("Espin_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("Espin_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("Bspin_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("Bspin_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("Bspin_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]
joined_EBspin_fac_cwt = {}
for v, _, _ in targets_EBspin:
    da, coi = concat_cwt_segments(ds_EBspin_fac_cwt_segs, v)
    if da is not None: joined_EBspin_fac_cwt[v] = (da, coi)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T20:00') + np.timedelta64(5, 'm')*n
#    for n in range(48)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EBspin), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets_EBspin):
#        if v not in joined_EBspin_fac_cwt: continue
#        da, coi = joined_EBspin_fac_cwt[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(np.nanmin(da.freq), np.nanmax(da.freq)),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_spin_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

# --- 連結（周波数を合わせて縦結合）
def concat_cwt_segments(dsets, var):
    das = [ds[var] for ds in dsets if var in ds]
    if not das: return None, None
    freqs = np.unique(np.concatenate([da.freq.values for da in das]))
    das = [da if np.array_equal(da.freq.values, freqs) else da.interp(freq=freqs) for da in das]
    pow_cat = xr.concat(das, dim="time").sortby("time").assign_coords(freq=("freq", freqs))
    coi_name = var.replace("_cwt", "_coi")
    coi_cat = xr.concat([ds[coi_name] for ds in dsets if coi_name in ds], dim="time").sortby("time") \
              if all(coi_name in ds for ds in dsets) else None
    return pow_cat, coi_cat

# --- 2帯域を同一axに描画
def plot_cwt_dualband_on_ax(ax,
                            da_hi, coi_hi,   # 高周波 (128Hz系)
                            da_lo, coi_lo,   # 低周波 (8Hz系)
                            t0, t1, minutes,
                            f_lo=(1e-2, 4.0),
                            f_hi=(4.0, 64.0),
                            zrange=(1e-6, 1e3),
                            cmap="turbo",
                            ylabel="", unit_right=""):

    if t1 is None:
        t1 = t0 + np.timedelta64(minutes, "m")

    # ---- データ切り出し ----
    dah  = da_hi.sel(time=slice(t0, t1)) if da_hi  is not None else None
    dal  = da_lo.sel(time=slice(t0, t1)) if da_lo  is not None else None
    coih = coi_hi.sel(time=slice(t0, t1)) if coi_hi is not None else None
    coil = coi_lo.sel(time=slice(t0, t1)) if coi_lo is not None else None

    if (dah is None or dah.time.size == 0) and (dal is None or dal.time.size == 0):
        return None, None

    def _prep(da, coi, fmin, fmax):
        if da is None or da.time.size == 0:
            return None, None, None
        da2 = da.sel(freq=slice(fmin, fmax))
        T = mdates.date2num(da2.time.values)
        F = da2.freq.values
        Z = da2.values.astype(float)
        if coi is not None:
            C = coi.values[:, None]
            Z = np.where(F[None, :] < C, np.nan, Z)
        Tm = np.tile(T, (F.size, 1)).T
        Fm = np.tile(F, (T.size, 1))
        return Tm, Fm, Z

    pcm_h = pcm_l = None

    # 低周波 (8 Hz 系)
    out_lo = _prep(dal, coil, *f_lo)
    if out_lo[0] is not None:
        Tm_l, Fm_l, Z_l = out_lo
        pcm_l = ax.pcolormesh(Tm_l, Fm_l, Z_l, shading="auto",
                              norm=LogNorm(vmin=zrange[0], vmax=zrange[1]),
                              cmap=cmap)

    # 高周波 (128 Hz 系)
    out_hi = _prep(dah, coih, *f_hi)
    if out_hi[0] is not None:
        Tm_h, Fm_h, Z_h = out_hi
        pcm_h = ax.pcolormesh(Tm_h, Fm_h, Z_h, shading="auto",
                              norm=LogNorm(vmin=zrange[0], vmax=zrange[1]),
                              cmap=cmap)

    # ---- 軸設定 ----
    ax.set_yscale("log")
    ax.set_ylim(f_lo[0], f_hi[1])
    ax.set_ylabel(f"{ylabel}\n[Hz]")

    # t0〜t1 をそのまま xlim に
    ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.minorticks_on()

    # ---- カラーバー ----
    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    pcm_main = pcm_h if pcm_h is not None else pcm_l
    cb = plt.colorbar(pcm_main, cax=cax)
    cb.set_label(unit_right)

    return (pcm_h, pcm_l), cb

In [ ]:
#targets_EB128_EB8 = [
#    ("E128_fac_x_cwt","E8_fac_x_cwt",  r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E128_fac_y_cwt","E8_fac_y_cwt",  r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E128_fac_z_cwt","E8_fac_z_cwt",  r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B128_fac_x_cwt","B8_fac_x_cwt",  r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B128_fac_y_cwt","B8_fac_y_cwt",  r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B128_fac_z_cwt","B8_fac_z_cwt",  r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
#]
#
#joined_EB128_EB8_fac_cwt = {}
#for v_hi, v_lo, _, _ in targets_EB128_EB8:
#    da_hi, coi_hi = concat_cwt_segments(ds_EB128_fac_cwt_segs, v_hi)
#    da_lo, coi_lo = concat_cwt_segments(ds_EB8_fac_cwt_segs,   v_lo)
#    if (da_hi is not None) or (da_lo is not None):
#        joined_EB128_EB8_fac_cwt[(v_hi, v_lo)] = (da_hi, coi_hi, da_lo, coi_lo)

In [ ]:
targets_EB128_EBspin = [
    ("E128_fac_x_cwt","Espin_fac_x_cwt",  r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E128_fac_y_cwt","Espin_fac_y_cwt",  r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E128_fac_z_cwt","Espin_fac_z_cwt",  r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B128_fac_x_cwt","Bspin_fac_x_cwt",  r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B128_fac_y_cwt","Bspin_fac_y_cwt",  r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B128_fac_z_cwt","Bspin_fac_z_cwt",  r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]

joined_EB128_EBspin_fac_cwt = {}
for v_hi, v_lo, _, _ in targets_EB128_EBspin:
    da_hi, coi_hi = concat_cwt_segments(ds_EB128_fac_cwt_segs,  v_hi)
    da_lo, coi_lo = concat_cwt_segments(ds_EBspin_fac_cwt_segs, v_lo)
    if (da_hi is not None) or (da_lo is not None):
        joined_EB128_EBspin_fac_cwt[(v_hi, v_lo)] = (da_hi, coi_hi, da_lo, coi_lo)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n
#    for n in range(12)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB128_EBspin), 1, figsize=(10, 10), sharex=True)
#    for ax, (v_hi, v_lo, ylab, unit) in zip(axes, targets_EB128_EBspin):
#        if (v_hi, v_lo) not in joined_EB128_EBspin_fac_cwt: continue
#        da_hi, coi_hi, da_lo, coi_lo = joined_EB128_EBspin_fac_cwt[(v_hi, v_lo)]
#        plot_cwt_dualband_on_ax(ax, da_hi, coi_hi, da_lo, coi_lo,
#                                t0=t0, minutes=5,
#                                f_lo=(1e-2, 0.1824), f_hi=(0.1824, 64.0),
#                                zrange=(1e-6, 1e3), cmap="turbo",
#                                ylabel=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

# 5分毎のPSDのMedianをplot

In [ ]:
da_Espin_fac_x_cwt  = joined_EBspin_fac_cwt["Espin_fac_x_cwt"]
da_Espin_fac_y_cwt  = joined_EBspin_fac_cwt["Espin_fac_y_cwt"]
da_Espin_fac_z_cwt  = joined_EBspin_fac_cwt["Espin_fac_z_cwt"]
da_Bspin_fac_x_cwt  = joined_EBspin_fac_cwt["Bspin_fac_x_cwt"]
da_Bspin_fac_y_cwt  = joined_EBspin_fac_cwt["Bspin_fac_y_cwt"]
da_Bspin_fac_z_cwt  = joined_EBspin_fac_cwt["Bspin_fac_z_cwt"]

da_E8_fac_x_cwt     = joined_EB8_fac_cwt["E8_fac_x_cwt"]
da_E8_fac_y_cwt     = joined_EB8_fac_cwt["E8_fac_y_cwt"]
da_E8_fac_z_cwt     = joined_EB8_fac_cwt["E8_fac_z_cwt"]
da_B8_fac_x_cwt     = joined_EB8_fac_cwt["B8_fac_x_cwt"]
da_B8_fac_y_cwt     = joined_EB8_fac_cwt["B8_fac_y_cwt"]
da_B8_fac_z_cwt     = joined_EB8_fac_cwt["B8_fac_z_cwt"]

da_E128_fac_x_cwt   = joined_EB128_fac_cwt["E128_fac_x_cwt"]
da_E128_fac_y_cwt   = joined_EB128_fac_cwt["E128_fac_y_cwt"]
da_E128_fac_z_cwt   = joined_EB128_fac_cwt["E128_fac_z_cwt"]
da_B128_fac_x_cwt   = joined_EB128_fac_cwt["B128_fac_x_cwt"]
da_B128_fac_y_cwt   = joined_EB128_fac_cwt["B128_fac_y_cwt"]
da_B128_fac_z_cwt   = joined_EB128_fac_cwt["B128_fac_z_cwt"]

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 20

# ---- 入力 ----
pairs_spin = [
    ("E_spin_FAC_x_cwt",    da_Espin_fac_x_cwt),
    ("E_spin_FAC_y_cwt",    da_Espin_fac_y_cwt),
    ("E_spin_FAC_z_cwt",    da_Espin_fac_z_cwt),
    ("B_spin_FAC_x_cwt",    da_Bspin_fac_x_cwt),
    ("B_spin_FAC_y_cwt",    da_Bspin_fac_y_cwt),
    ("B_spin_FAC_z_cwt",    da_Bspin_fac_z_cwt),
]

pairs_8 = [
    ("E_8_FAC_x_cwt",   da_E8_fac_x_cwt),
    ("E_8_FAC_y_cwt",   da_E8_fac_y_cwt),
    ("E_8_FAC_z_cwt",   da_E8_fac_z_cwt),
    ("B_8_FAC_x_cwt",   da_B8_fac_x_cwt),
    ("B_8_FAC_y_cwt",   da_B8_fac_y_cwt),
    ("B_8_FAC_z_cwt",   da_B8_fac_z_cwt),
]

pairs_128 = [
    ("E_128_FAC_x_cwt", da_E128_fac_x_cwt),
    ("E_128_FAC_y_cwt", da_E128_fac_y_cwt),
    ("E_128_FAC_z_cwt", da_E128_fac_z_cwt),
    ("B_128_FAC_x_cwt", da_B128_fac_x_cwt),
    ("B_128_FAC_y_cwt", da_B128_fac_y_cwt),
    ("B_128_FAC_z_cwt", da_B128_fac_z_cwt),
]

t_all_start = np.datetime64('2022-09-01T20:00:00')
t_all_end   = np.datetime64('2022-09-02T00:00:00')
step        = np.timedelta64(5, 'm')  # 5分
outdir      = (
    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330/wavelet_PSD/5min_PSD"
)
os.makedirs(outdir, exist_ok=True)

# 事前に CWT 本体のみ取り出し、timeでソート
pairs_spin_sorted = []
for name, da in pairs_spin:
    if isinstance(da, tuple):
        da = da[0]
    if isinstance(da, xr.DataArray):
        pairs_spin_sorted.append((name, da.sortby('time')))

pairs_8_sorted = []
for name, da in pairs_8:
    if isinstance(da, tuple):
        da = da[0]
    if isinstance(da, xr.DataArray):
        pairs_8_sorted.append((name, da.sortby('time')))

pairs_128_sorted = []
for name, da in pairs_128:
    if isinstance(da, tuple):
        da = da[0]
    if isinstance(da, xr.DataArray):
        pairs_128_sorted.append((name, da.sortby('time')))

def compute_median_dict(pairs_sorted, t0, t1):
    d = {}
    for name, da in pairs_sorted:
        sub = da.sel(time=slice(t0, t1))
        if sub.sizes.get('time', 0) == 0:
            continue
        if np.iscomplexobj(sub.data):
            sub = (sub.real**2 + sub.imag**2)
        d[name] = sub.median(dim='time', skipna=True)  # (freq,)
    return d

def plot_median_dict(mdict_1, mdict_2, mdict_3, t0, t1, outdir=None):
    fig, ax = plt.subplots(figsize=(8, 8))

    color_map = {
        ('E', 'x'): 'blue',      # Ex
        ('E', 'y'): 'orange',    # Ey
        ('E', 'z'): 'brown',     # Ez
        ('B', 'x'): 'green',     # Bx
        ('B', 'y'): 'red',       # By
        ('B', 'z'): 'purple',    # Bz
    }

    for name, med in mdict_1.items():
        if name.split('_')[3] == 'z':
            continue
        prefix  = name.split('_')[0]
        freq    = name.split('_')[1]
        comp    = name.split('_')[3]
        coor    = name.split('_')[2]
        label  = f"${prefix}_{comp}$ ({coor})"
        #label   = f"${prefix}_{comp} $"# ({coor}, {freq})"
        col     = color_map.get((prefix, comp), 'gray')
        ax.loglog(med['freq'], med, label=label, lw=1, color=col)

    #for name, med in mdict_2.items():
    #    prefix  = name.split('_')[0]
    #    freq    = name.split('_')[1]
    #    comp    = name.split('_')[3]
    #    coor    = name.split('_')[2]
    #    label  = f"${prefix}_{comp}$ ({coor}, {freq} Hz)"
    #    ax.loglog(med['freq'], med, label=label)

    for name, med in mdict_3.items():
        if name.split('_')[3] == 'z':
            continue
        prefix  = name.split('_')[0]
        freq    = name.split('_')[1]
        comp    = name.split('_')[3]
        coor    = name.split('_')[2]
        label  = f"${prefix}_{comp}$ ({coor}, {freq} Hz)"
        col     = color_map.get((prefix, comp), 'gray')
        ax.loglog(med['freq'][med['freq']>0.3648/2.], med[med['freq']>0.3648/2.], lw=1, color=col)
    
    ax.axvline(0.3648/2., c='k', lw=1)

    ax.minorticks_on()
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Median PSD')
    ax.set_title(f"Median {str(t0)[11:]}–{str(t1)[11:]}\n (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)")
    ax.grid(True, which='both', ls=':')
    ax.set_yticks([1E-8, 1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
    ax.legend(ncol=2, fontsize=15)
    ax.set_xlim(1e-2, 64)
    ax.set_ylim(1e-8, 1e4)
    plt.tight_layout()

    if outdir and os.path.isdir(outdir):
        fn = f"median_bs_E_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
        fig.savefig(os.path.join(outdir, fn), dpi=300, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)
##
### ---- 5分窓でループ ----
##t_starts = np.arange(t_all_start, t_all_end, step)  # 21:00, 21:05, ..., 23:25
##for t0 in t_starts:
##    t1 = t0 + step
##    mdict_spin  = compute_median_dict(pairs_spin_sorted, t0, t1)
##    mdict_8     = compute_median_dict(pairs_8_sorted, t0, t1)
##    mdict_128   = compute_median_dict(pairs_128_sorted, t0, t1)
##    if not mdict_spin and not mdict_8 and not mdict_128:  # その窓でデータ無し
##        continue
##    plot_median_dict(mdict_spin, mdict_8, mdict_128, t0, t1, outdir=outdir)
#
t_starts    = np.datetime64('2022-09-01T23:05:45')
t_ends      = np.datetime64('2022-09-01T23:08:15')
mdict_spin  = compute_median_dict(pairs_spin_sorted, t_starts, t_ends)
mdict_128   = compute_median_dict(pairs_128_sorted, t_starts, t_ends)
plot_median_dict(mdict_spin, None, mdict_128, t_starts, t_ends, outdir)

In [ ]:
mu_0 = 4.*np.pi*1E-7

Ex_fac  = xr.concat([ds_EB128_fac_segs[0]['E128_fac_x'], ds_EB128_fac_segs[1]['E128_fac_x'], ds_EB128_fac_segs[2]['E128_fac_x']], dim='time')
Ey_fac  = xr.concat([ds_EB128_fac_segs[0]['E128_fac_y'], ds_EB128_fac_segs[1]['E128_fac_y'], ds_EB128_fac_segs[2]['E128_fac_y']], dim='time')
Ez_fac  = xr.concat([ds_EB128_fac_segs[0]['E128_fac_z'], ds_EB128_fac_segs[1]['E128_fac_z'], ds_EB128_fac_segs[2]['E128_fac_z']], dim='time')
Bx_fac  = xr.concat([ds_EB128_fac_segs[0]['B128_fac_x'], ds_EB128_fac_segs[1]['B128_fac_x'], ds_EB128_fac_segs[2]['B128_fac_x']], dim='time')
By_fac  = xr.concat([ds_EB128_fac_segs[0]['B128_fac_y'], ds_EB128_fac_segs[1]['B128_fac_y'], ds_EB128_fac_segs[2]['B128_fac_y']], dim='time')
Bz_fac  = xr.concat([ds_EB128_fac_segs[0]['B128_fac_z'], ds_EB128_fac_segs[1]['B128_fac_z'], ds_EB128_fac_segs[2]['B128_fac_z']], dim='time')

S_para  = (Ex_fac * By_fac - Ey_fac * Bx_fac) / mu_0 * 1E-12

S_para_toroidal = Ex_fac * By_fac / mu_0 * 1E-12
S_para_poloidal = - Ey_fac * Bx_fac / mu_0 * 1E-12

print(S_para)
print(S_para_toroidal)
print(S_para_poloidal)

In [ ]:
Vph_toroidal    = np.abs(Ey_fac / Bx_fac) * 1E6 # [m/s]
Vph_poloidal    = np.abs(Ex_fac / By_fac) * 1E6 # [m/s]

Vph_perp_comp   = np.sqrt((Ex_fac**2E0 + Ey_fac**2E0) / (Bx_fac**2E0 + By_fac**2E0)) * 1E6 # [m/s]

In [ ]:
path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330/wavelet_PSD"
)
os.makedirs(path_base_save_plot, exist_ok=True)

# 22:30:45~22:32:45, 22:48:30~22:51:00, 23:05:45~23:08:15

In [ ]:
#import matplotlib as mpl
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 20
#
#time_windows = [
#    (np.datetime64('2022-09-01T22:30:45'),
#     np.datetime64('2022-09-01T22:32:45')),
#
#    (np.datetime64('2022-09-01T22:48:30'),
#     np.datetime64('2022-09-01T22:51:00')),
#
#    (np.datetime64('2022-09-01T23:05:45'),
#     np.datetime64('2022-09-01T23:08:15')),
#]
#
#def add_panel_label(ax, label, x=-0.10, y=0.95):
#    ax.text(x, y, label, transform=ax.transAxes,
#            ha='right', va='bottom', clip_on=False)
#
#for t0, t1 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB128_EBspin)+2, 1, figsize=(14, 18), sharex=True)
#
#    # --- dualband CWT 部分 ---
#    for ax, (v_hi, v_lo, ylab, unit) in zip(axes, targets_EB128_EBspin):
#        if (v_hi, v_lo) not in joined_EB128_EBspin_fac_cwt: continue
#        da_hi, coi_hi, da_lo, coi_lo = joined_EB128_EBspin_fac_cwt[(v_hi, v_lo)]
#        plot_cwt_dualband_on_ax(
#            ax, da_hi, coi_hi, da_lo, coi_lo,
#            t0=t0, t1=t1, minutes=None,
#            f_lo=(1e-2, 0.1824), f_hi=(0.1824, 64.0),
#            zrange=(1e-6, 1e3), cmap="turbo",
#            ylabel=ylab, unit_right=unit
#        )
#        ax.axhline(0.1824, c='k', lw=2)
#
#    S_para_window = S_para.sel(time=slice(t0, t1))
#    S_para_toroidal_window = S_para_toroidal.sel(time=slice(t0, t1))
#    S_para_poloidal_window = S_para_poloidal.sel(time=slice(t0, t1))
#
#    axes[len(targets_EB128_EBspin)].plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5, label='total')
#    axes[len(targets_EB128_EBspin)].set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    axes[len(targets_EB128_EBspin)].minorticks_on()
#    axes[len(targets_EB128_EBspin)].grid(which='both', alpha=0.5)
#
##    axes[len(targets_EB128_EBspin)+1].plot(S_para_poloidal_window.time, S_para_poloidal_window.data*1E3, c='green', #linewidth=0.5, label='poloidal')
##    axes[len(targets_EB128_EBspin)+1].plot(S_para_toroidal_window.time, S_para_toroidal_window.data*1E3, c='orange', #linewidth=0.5, label='toroidal')
##    axes[len(targets_EB128_EBspin)+1].set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
##    axes[len(targets_EB128_EBspin)+1].minorticks_on()
##    axes[len(targets_EB128_EBspin)+1].grid(which='both', alpha=0.5)
##    axes[len(targets_EB128_EBspin)+1].legend(ncol=5, loc='lower left', fontsize=10)
#
#    Vph_poloidal_window = Vph_poloidal.sel(time=slice(t0, t1))
#    Vph_toroidal_window = Vph_toroidal.sel(time=slice(t0, t1))
#    Vph_perp_window     = Vph_perp_comp.sel(time=slice(t0, t1))
#    VA_window = ds_velocity_ms_perp['Alfven_speed'].sel(time=slice(t0, t1))
#    print(Vph_perp_window)
#
#    axes[len(targets_EB128_EBspin)+1].plot(Vph_perp_window.time, Vph_perp_window.data*1E-3, c='k', lw=0.5, linestyle='solid', label=r'$|\mathbf{E}_{\perp}| / |\mathbf{B}_{\perp}|$')
#    axes[len(targets_EB128_EBspin)+1].plot(VA_window.time, VA_window.data*1E-3, c='blue', lw=2, linestyle='dotted', label=r'$v_{\mathrm{A}}$')
#    axes[len(targets_EB128_EBspin)+1].set_ylabel(r'Velocity' + '\n' + r'[$\mathrm{km/s}$]')
#    axes[len(targets_EB128_EBspin)+1].set_ylim(ymin=1E2, ymax=1E5)
#    axes[len(targets_EB128_EBspin)+1].set_yscale('log')
#    axes[len(targets_EB128_EBspin)+1].minorticks_on()
#    axes[len(targets_EB128_EBspin)+1].grid(which='both', alpha=0.5)
#    axes[len(targets_EB128_EBspin)+1].legend(ncol=2, loc='lower left', fontsize=15)
#
##    axes[len(targets_EB128_EBspin)+2].plot(Vph_poloidal_window.time, Vph_poloidal_window.data*1E-3, c='green', lw=0.5, ##linestyle='solid', label='poloidal')
##    axes[len(targets_EB128_EBspin)+2].plot(Vph_toroidal_window.time, Vph_toroidal_window.data*1E-3, c='orange', lw=0.5, ##linestyle='solid', label='toroidal')
##    axes[len(targets_EB128_EBspin)+2].plot(VA_window.time, VA_window.data*1E-3, c='blue', lw=2, linestyle='dotted', ##label=r'$v_{\mathrm{A}}$')
##    axes[len(targets_EB128_EBspin)+2].set_ylabel(r'Velocity' + '\n' + r'[$\mathrm{km/s}$]')
##    axes[len(targets_EB128_EBspin)+2].set_ylim(ymin=1E2, ymax=1E5)
##    axes[len(targets_EB128_EBspin)+2].set_yscale('log')
##    axes[len(targets_EB128_EBspin)+2].minorticks_on()
##    axes[len(targets_EB128_EBspin)+2].grid(which='both', alpha=0.5)
##    axes[len(targets_EB128_EBspin)+2].legend(ncol=5, loc='lower left', fontsize=10)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#    fig.subplots_adjust(hspace=0.1)
#
#    add_panel_label(axes[0], '(1)')
#    add_panel_label(axes[1], '(2)')
#    add_panel_label(axes[2], '(3)')
#    add_panel_label(axes[3], '(4)')
#    add_panel_label(axes[4], '(5)')
#    add_panel_label(axes[5], '(6)')
#    add_panel_label(axes[6], '(7)')
#    add_panel_label(axes[7], '(8)')
#    #add_panel_label(axes[8], '(9)')
#
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_{fn_time}_for_figure.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

# E/B plot

- Poloidal components: $B_{x}$, $E_{y}$
- Toroidal components: $B_{y}$, $E_{x}$

In [ ]:
da_zeros_spin       = xr.zeros_like(joined_EBspin_fac_cwt['Espin_fac_x_cwt'][0])
time_interp_base    = (joined_EB128_fac_cwt['E128_fac_x_cwt'][0]).time
da_zeros_spin       = da_zeros_spin.interp(time=time_interp_base).sortby('freq')

da_E_spin_fac_x_interp  = (joined_EBspin_fac_cwt['Espin_fac_x_cwt'][0]).interp(time=time_interp_base).sortby('freq')
da_E_spin_fac_y_interp  = (joined_EBspin_fac_cwt['Espin_fac_y_cwt'][0]).interp(time=time_interp_base).sortby('freq')
da_B_spin_fac_x_interp  = (joined_EBspin_fac_cwt['Bspin_fac_x_cwt'][0]).interp(time=time_interp_base).sortby('freq')
da_B_spin_fac_y_interp  = (joined_EBspin_fac_cwt['Bspin_fac_y_cwt'][0]).interp(time=time_interp_base).sortby('freq')

ds_EBspin_fac_cwt_toroidal  = xr.Dataset({
    'Espin_fac_x_cwt':  da_E_spin_fac_x_interp,
    'Espin_fac_y_cwt':  da_zeros_spin,
    'Espin_fac_z_cwt':  da_zeros_spin,
    'Bspin_fac_x_cwt':  da_zeros_spin,
    'Bspin_fac_y_cwt':  da_B_spin_fac_y_interp,
    'Bspin_fac_z_cwt':  da_zeros_spin,
})

ds_EBspin_fac_cwt_poloidal  = xr.Dataset({
    'Espin_fac_x_cwt':  da_zeros_spin,
    'Espin_fac_y_cwt':  da_E_spin_fac_y_interp,
    'Espin_fac_z_cwt':  da_zeros_spin,
    'Bspin_fac_x_cwt':  da_B_spin_fac_x_interp,
    'Bspin_fac_y_cwt':  da_zeros_spin,
    'Bspin_fac_z_cwt':  da_zeros_spin,
})

ds_EBspin_fac_cwt_perp      = xr.Dataset({
    'Espin_fac_x_cwt':  da_E_spin_fac_x_interp,
    'Espin_fac_y_cwt':  da_E_spin_fac_y_interp,
    'Espin_fac_z_cwt':  da_zeros_spin,
    'Bspin_fac_x_cwt':  da_B_spin_fac_x_interp,
    'Bspin_fac_y_cwt':  da_B_spin_fac_y_interp,
    'Bspin_fac_z_cwt':  da_zeros_spin,
})

print(ds_EBspin_fac_cwt_toroidal)
print('')
print(ds_EBspin_fac_cwt_poloidal)
print('')
print(ds_EBspin_fac_cwt_perp)

In [ ]:
da_zeros_128   = xr.zeros_like(joined_EB128_fac_cwt['E128_fac_x_cwt'][0]).sortby('freq')

da_E_128_fac_x_interp  = (joined_EB128_fac_cwt['E128_fac_x_cwt'][0]).sortby('freq')
da_E_128_fac_y_interp  = (joined_EB128_fac_cwt['E128_fac_y_cwt'][0]).sortby('freq')
da_B_128_fac_x_interp  = (joined_EB128_fac_cwt['B128_fac_x_cwt'][0]).sortby('freq')
da_B_128_fac_y_interp  = (joined_EB128_fac_cwt['B128_fac_y_cwt'][0]).sortby('freq')

ds_EB128_fac_cwt_toroidal  = xr.Dataset({
    'E128_fac_x_cwt':  da_E_128_fac_x_interp,
    'E128_fac_y_cwt':  da_zeros_128,
    'E128_fac_z_cwt':  da_zeros_128,
    'B128_fac_x_cwt':  da_zeros_128,
    'B128_fac_y_cwt':  da_B_128_fac_y_interp,
    'B128_fac_z_cwt':  da_zeros_128,
})

ds_EB128_fac_cwt_poloidal  = xr.Dataset({
    'E128_fac_x_cwt':  da_zeros_128,
    'E128_fac_y_cwt':  da_E_128_fac_y_interp,
    'E128_fac_z_cwt':  da_zeros_128,
    'B128_fac_x_cwt':  da_B_128_fac_x_interp,
    'B128_fac_y_cwt':  da_zeros_128,
    'B128_fac_z_cwt':  da_zeros_128,
})

ds_EB128_fac_cwt_perp      = xr.Dataset({
    'E128_fac_x_cwt':  da_E_128_fac_x_interp,
    'E128_fac_y_cwt':  da_E_128_fac_y_interp,
    'E128_fac_z_cwt':  da_zeros_128,
    'B128_fac_x_cwt':  da_B_128_fac_x_interp,
    'B128_fac_y_cwt':  da_B_128_fac_y_interp,
    'B128_fac_z_cwt':  da_zeros_128,
})

print(ds_EB128_fac_cwt_toroidal)
print('')
print(ds_EB128_fac_cwt_poloidal)
print('')
print(ds_EB128_fac_cwt_perp)

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import module_handmade.psd_plotter_themis_xarray as psdptx
import importlib
importlib.reload(psdptx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction           = 'perp'
dsets_spin_clean    = [ds_EBspin_fac_cwt_perp]
dsets_128           = [ds_EB128_fac_cwt_perp]
ds_velocity_ms      = ds_velocity_ms_perp

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdptx.build_data_dict_xr(
    dsets_spin_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit


# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdptx.plot_freq_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import module_handmade.psd_plotter_themis_xarray as psdptx
import importlib
importlib.reload(psdptx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction           = 'poloidal'
dsets_spin_clean    = [ds_EBspin_fac_cwt_poloidal]
dsets_128           = [ds_EB128_fac_cwt_poloidal]
ds_velocity_ms      = ds_velocity_ms_poloidal

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdptx.build_data_dict_xr(
    dsets_spin_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit


# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdptx.plot_freq_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import module_handmade.psd_plotter_themis_xarray as psdptx
import importlib
importlib.reload(psdptx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction           = 'toroidal'
dsets_spin_clean    = [ds_EBspin_fac_cwt_toroidal]
dsets_128           = [ds_EB128_fac_cwt_toroidal]
ds_velocity_ms      = ds_velocity_ms_toroidal

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdptx.build_data_dict_xr(
    dsets_spin_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit


# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdptx.plot_freq_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import module_handmade.psd_plotter_themis_xarray as psdptx
import importlib
importlib.reload(psdptx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

direction           = 'perp'
dsets_spin          = [ds_EBspin_fac_cwt_perp]
dsets_128           = [ds_EB128_fac_cwt_perp]
ds_velocity_ms      = ds_velocity_ms_perp
S_para_select       = S_para

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdptx.build_data_dict_xr(
    dsets_spin, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig, fit_results = psdptx.plot_k_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt, n_bins=30, fit_range=(3, 30))
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)
        return fit_results

#time_range = ['2022-09-01T22:30:45', '2022-09-01T22:32:45']
#time_range = ['2022-09-01T22:29:00', '2022-09-01T22:34:00']
#time_range = ['2022-09-01T22:48:30', '2022-09-01T22:51:30']
time_range = ['2022-09-01T23:05:45', '2022-09-01T23:08:15']

t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para_select.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    #ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
    #            fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    ax_0.plot(df_results.index, df_results['kappa_B'], marker='.', c='blue', label=r'$\kappa_{\mathrm{B}}$', lw=2)
    ax_0.axhline(7./3., lw=2, linestyle='dotted', c='blue', label=r'$\kappa_{\mathrm{B}}$ in KAW cascade')
    
    # kappa_Eのプロット（エラーバー付き）
    #ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
    #            fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    ax_0.plot(df_results.index, df_results['kappa_E'], marker='.', c='orange', label=r'$\kappa_{\mathrm{E}}$', lw=2)
    ax_0.axhline(1./3., lw=2, linestyle='dotted', c='orange', label=r'$\kappa_{\mathrm{E}}$ in KAW cascade')
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)' + f' ({direction})')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend(loc='best', ncol=4, fontsize=12)
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import module_handmade.psd_plotter_themis_xarray as psdptx
import importlib
importlib.reload(psdptx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

direction           = 'poloidal'
dsets_spin          = [ds_EBspin_fac_cwt_poloidal]
dsets_128           = [ds_EB128_fac_cwt_poloidal]
ds_velocity_ms      = ds_velocity_ms_poloidal
S_para_select       = S_para_poloidal

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdptx.build_data_dict_xr(
    dsets_spin, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig, fit_results = psdptx.plot_k_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt, n_bins=30)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)
        return fit_results

#time_range = ['2022-09-01T22:29:00', '2022-09-01T22:34:00']
#time_range = ['2022-09-01T22:47:30', '2022-09-01T22:52:30']
time_range = ['2022-09-01T23:04:00', '2022-09-01T23:09:00']

t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para_select.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)' + f'({direction})')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import module_handmade.psd_plotter_themis_xarray as psdptx
import importlib
importlib.reload(psdptx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

direction           = 'toroidal'
dsets_spin          = [ds_EBspin_fac_cwt_toroidal]
dsets_128           = [ds_EB128_fac_cwt_toroidal]
ds_velocity_ms      = ds_velocity_ms_toroidal
S_para_select       = S_para_toroidal

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdptx.build_data_dict_xr(
    dsets_spin, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig, fit_results = psdptx.plot_k_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt, n_bins=30)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)
        return fit_results

#time_range = ['2022-09-01T22:29:00', '2022-09-01T22:34:00']
#time_range = ['2022-09-01T22:47:30', '2022-09-01T22:52:30']
time_range = ['2022-09-01T23:04:00', '2022-09-01T23:09:00']

t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para_select.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)' + f'({direction})')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")